# Code to generate reference 3D ellipsoid STL files for printing/measurements

In [1]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pyvista as pv
import trimesh
from copy import deepcopy
import pymeshfix as mf
%matplotlib widget

## Now generate different ellipsoids

In [2]:
#Use pyvista to generate an oblate spheroid. Increase u_res, v_res, w_res will increase file size.
#https://docs.pyvista.org/api/utilities/_autosummary/pyvista.parametricellipsoid
ellipsoid_surf = pv.ParametricEllipsoid(xradius = 30, yradius=30, zradius=12)

#triangulate
ellipsoid_surf_x2y2z1 = ellipsoid_surf.triangulate()



## Plot the results

In [3]:
# pl = pv.Plotter(shape=(1, 2))
# pl.subplot(0, 1)
# _ = pl.add_mesh(ellipsoid_surf ,color='blue')
# pl.show()

## Write stuff out

In [4]:
# Convert pyvista polydata to trimesh and export
ellipsoid_tm = trimesh.Trimesh(vertices=ellipsoid_surf.points, faces=ellipsoid_surf.faces.reshape(-1, 4)[:, 1:])

# Export to STL files
ellipsoid_tm.export('ellipsoid_surf.stl');


# Load/repair mesh, remove sphere from input shape.

In [5]:
def load_and_repair(path):
    mesh = trimesh.load(path, force="mesh")
    
    #needs to be watertight but also check face winding and volume to fix.
    if not mesh.is_watertight:
        v, f = mf.clean_from_arrays(mesh.vertices, mesh.faces)
        mesh = trimesh.Trimesh(v, f)
    
    if not mesh.is_winding_consistent:
        trimesh.repair.fix_winding(mesh)

    if mesh.volume < 0:
        mesh.invert()    
    
    return mesh
 
 
def subtract_sphere(mesh, center, radius):
    cavity = trimesh.creation.icosphere(subdivisions=4, radius=radius)
    cavity.apply_translation(center)
    return trimesh.boolean.difference([mesh, cavity], engine="manifold")

# Generate a jar lid-style threaded collar to join two hailstone halves together.

In [6]:
def generate_thread_collars():
    """Generates plug + socket printable thread collars using cq_warehouse's
    PlasticBottleThread (ASTM D2911) - a coarse-pitch, generous-clearance
    profile designed specifically for FDM-printed jars and bottle caps."""
    import cadquery as cq
    from cq_warehouse.thread import PlasticBottleThread
    
    # ASTM plastic bottle thread callout, e.g. "M38SP444" ~ 38 mm nominal dia.
    # Pick a size close to the flange ring diameter you want (must stay inside
    # the hailstone's silhouette at the cut plane). See ASTM D2911 tables /
    # cq_warehouse docs for the full size list.
    THREAD_SIZE          = "M24SP400" #standard that corresponds to the diameter of the screw.
                                      #24 is the diameter there. 400 corresponds to the finish;
                                      #different finishes have different available diameters.
                                      # see thread.py in the cq_warehouse package.
    THREAD_LENGTH         = 5.0    # mm, axial engagement (collar height)
    MANUFACTURING_COMP    = 0.20    # mm print clearance - start at 0.2, tune from test prints
 
    plug_thread = PlasticBottleThread(
        size=THREAD_SIZE,
        external=True,
        manufacturingCompensation=MANUFACTURING_COMP,
    )
    socket_thread = PlasticBottleThread(
        size=THREAD_SIZE,
        external=False,
        manufacturingCompensation=MANUFACTURING_COMP,
    )
 
    # Sanity check before cutting geometry - for an external thread root_radius
    # should be the smaller of the two (valley between ridges), and for an
    # internal thread root_radius should be the larger of the two (bore wall).
    # If either assertion fails, the root/apex convention is flipped from what's
    # assumed below - print the actual values and re-check before proceeding.
    print("plug   root/apex:", plug_thread.root_radius, plug_thread.apex_radius)
    print("socket root/apex:", socket_thread.root_radius, socket_thread.apex_radius)
    assert plug_thread.root_radius < plug_thread.apex_radius
    assert socket_thread.root_radius > socket_thread.apex_radius
 
    WALL_THICKNESS = 2.0  # mm, extra material around the socket bore
 
    # Solid cores so each collar is one printable solid, not a free
    # floating spiral ridge. Plug core is built out to the thread root
    # (the valleys), then the ridge solid fuses on top of it. Socket core
    # is a bore cut to the thread root (the bore wall), so the internal
    # ridge solid fuses flush onto its inner wall.
    plug_core = (
        cq.Workplane("XY")
        .circle(plug_thread.root_radius)
        .extrude(THREAD_LENGTH)
    )
    socket_core = (
        cq.Workplane("XY")
        .circle(socket_thread.root_radius + WALL_THICKNESS)
        .circle(socket_thread.root_radius)
        .extrude(THREAD_LENGTH)
    )
 
    plug_collar = plug_thread.fuse(plug_core.val())
    socket_collar = socket_thread.fuse(socket_core.val())
 
    cq.exporters.export(plug_collar, "plug_thread_collar.stl", tolerance=0.001, angularTolerance=0.005)
    cq.exporters.export(socket_collar, "socket_thread_collar.stl", tolerance=0.001, angularTolerance=0.005)

# Given the whole hailstone mesh, split it in half and add the threaded collar.

In [ ]:
def attach_collars(mesh, mesh_center):

    CUT_PLANE_NORMAL     = np.array([0.0, 0.0, 1.0])    # "up" direction for the cut
    CUT_PLANE_POINT      = mesh_center                # slice through the hailstone center

    """Slices the (still solid, not-yet-hollowed) mesh into top and bottom
    halves and attaches the thread collars to each. The instrumentation
    cavity is carved out afterward, per half, in subtract_sphere_from_halves()."""
    top = trimesh.intersections.slice_mesh_plane(
        mesh, plane_normal=CUT_PLANE_NORMAL, plane_origin=CUT_PLANE_POINT, cap=True
    )
    bottom = trimesh.intersections.slice_mesh_plane(
        mesh, plane_normal=-CUT_PLANE_NORMAL, plane_origin=CUT_PLANE_POINT, cap=True
    )
 
    #load plug collar via trimesh, so we can do stuff with it afterward.
    # process=False turns off automatic cleanup/merging so we can do that manually first
    plug_collar = trimesh.load("plug_thread_collar.stl", process=False)
    plug_collar.vertices = np.round(plug_collar.vertices, decimals=2) # merge points within ~0.001mm of each other
                                    #to match my output tolerance when exporting from cq
    #now we can do the cleanup
    plug_collar.merge_vertices()  
    plug_collar.fill_holes()
    plug_collar.process() #automatic cleanup now.
    #confirm it's watertight - had occasional problems with this
    print("watertight:", plug_collar.is_watertight, "is_volume:", plug_collar.is_volume)
    if not plug_collar.is_watertight:
        print ('fixing plug not watertight issue')
        v, f = mf.clean_from_arrays(plug_collar.vertices, plug_collar.faces)
        plug_collar = trimesh.Trimesh(v, f)

    socket_collar = trimesh.load("socket_thread_collar.stl")
    print("watertight:", socket_collar.is_watertight, "is_volume:", socket_collar.is_volume)
    if not socket_collar.is_watertight:
        print ('fixing socket not watertight issue')
        v, f = mf.clean_from_arrays(socket_collar.vertices, socket_collar.faces)
        socket_collar = trimesh.Trimesh(v, f)
 
    # Align + position the collars so their axis matches CUT_PLANE_NORMAL
    # and their base sits flush with the cut plane, centered on the
    # cavity center. Adjust translation/rotation for your geometry -
    # exact offsets depend on how far the cavity opening's rim sits from
    # CUT_PLANE_POINT.
    plug_collar.apply_translation(mesh_center)
    socket_collar.apply_translation(mesh_center)
 
    bottom = trimesh.boolean.union([bottom, plug_collar], engine="manifold")
    top = trimesh.boolean.difference([top, socket_collar], engine="manifold")
 
 
    return top, bottom

# Final mesh repair so watertight.

In [8]:
def repair_and_export(mesh, path):
    if not mesh.is_watertight:
        v, f = mf.clean_from_arrays(mesh.vertices, mesh.faces)
        mesh = trimesh.Trimesh(v, f)
    mesh.export(path)
    print(f"{path}: watertight={mesh.is_watertight}, volume={mesh.volume:.1f} mm^3")

# Now, run it all!

In [9]:
mesh = load_and_repair('ellipsoid_surf.stl')

CAVITY_CENTER        = np.array([0.0, 0.0, 0.0])   # mm, in model coords
CAVITY_RADIUS        = 9.0                        # mm - sized to instrument

generate_thread_collars()
top, bottom = attach_collars(mesh, CAVITY_CENTER)

"""Carves the instrumentation cavity out of each already-collared half
independently, using the FULL sphere (not pre-clipped) on each - since
each half only occupies material on one side of the cut plane already,
this only removes the portion of the sphere that actually overlaps that
half."""
top = subtract_sphere(top, CAVITY_CENTER, CAVITY_RADIUS)
bottom = subtract_sphere(bottom, CAVITY_CENTER, CAVITY_RADIUS)

repair_and_export(top,'ellipsoid_surf_x2y2z1_top.stl')
repair_and_export(bottom, 'ellipsoid_surf_x2y2z1_bottom.stl')

plug   root/apex: 10.465 11.535
socket root/apex: 12.139999999999999 11.069999999999999


TypeError: Trimesh.merge_vertices() got an unexpected keyword argument 'digits'

# Plot the things to see if they look reasonable

In [ ]:
#look for holes
pv_mesh = pv.wrap(plug_collar)
edges = pv_mesh.extract_feature_edges(boundary_edges=True, feature_edges=False, manifold_edges=False)
plotter = pv.Plotter()
plotter.add_mesh(pv_mesh, opacity=0.5)
plotter.add_mesh(edges, color='red', line_width=3)
plotter.show()

In [ ]:
# Show the mesh (opens in a window)
top.show()

In [ ]:
bottom.show()